In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
from langchain.tools import tool
#creating tools
@tool
def square_root(x: float) -> float:
    """Calculate the square root of a number"""
    return x ** 0.5

@tool
def square(x: float) -> float:
    """Calculate the square of a number"""
    return x ** 2

In [4]:
from langchain.agents import create_agent
from langchain_groq import ChatGroq
#create subagents and call tools
model=ChatGroq(model="openai/gpt-oss-20b")
subagent_1 = create_agent(
    model=model,
    tools=[square_root]
)

subagent_2 = create_agent(
    model=model,
    tools=[square]
)

In [5]:
#calling subagents as tools inmain agent
from langchain.messages import HumanMessage

@tool
def call_subagent_1(x: float) -> float:
    """Call subagent 1 in order to calculate the square root of a number"""
    response = subagent_1.invoke({"messages": [HumanMessage(content=f"Calculate the square root of {x}")]})
    return response["messages"][-1].content

@tool
def call_subagent_2(x: float) -> float:
    """Call subagent 2 in order to calculate the square of a number"""
    response = subagent_2.invoke({"messages": [HumanMessage(content=f"Calculate the square of {x}")]})
    return response["messages"][-1].content

## Creating the main agent

main_agent = create_agent(
    model=model,
    tools=[call_subagent_1, call_subagent_2],
    system_prompt="You are a helpful assistant who can call subagents to calculate the square root or square of a number.")


In [6]:
question = "What is the square root of 81?"

response = main_agent.invoke({"messages": [HumanMessage(content=question)]})

from pprint import pprint
pprint(response)

{'messages': [HumanMessage(content='What is the square root of 81?', additional_kwargs={}, response_metadata={}, id='580478cf-a17f-4a6e-b37e-91fa8a7d4915'),
              AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to call subagent 1 to calculate sqrt of 81.', 'tool_calls': [{'id': 'fc_5207a335-2ee0-4a70-ba12-f66c606c6ea4', 'function': {'arguments': '{"x":81}', 'name': 'call_subagent_1'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 42, 'prompt_tokens': 198, 'total_tokens': 240, 'completion_time': 0.043995516, 'completion_tokens_details': {'reasoning_tokens': 16}, 'prompt_time': 0.011006581, 'prompt_tokens_details': None, 'queue_time': 0.521655182, 'total_time': 0.055002097}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_e189667b30', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a1c9bc-c3e0-756d-a43e-f412b6fe03ca-0', tool_calls=[{'name': 